# 📹 Video 4: Association Rules dengan Algoritma Apriori

**Mata Kuliah**: Data Mining (21TIF604)  
**Universitas Islam Nahdlatul Ulama Jepara**  
**Dosen**: Ir. Adi Sucipto, M.Kom

---

### Tujuan Video Ini:
1. Memahami konsep Association Rules dan algoritma Apriori
2. Menyiapkan data dalam format transaksi (basket)
3. Menemukan frequent itemsets dan aturan asosiasi
4. Menginterpretasikan metrik: Support, Confidence, Lift
5. Mengaitkan hasil dengan konteks bisnis retail (bundling produk)

---

### Konteks Bisnis (Renstra):
Association Rules menemukan pola "pelanggan yang membeli produk A juga cenderung membeli produk B".  
Hasil ini bisa digunakan untuk: **bundling produk**, **penataan layout toko**, dan **rekomendasi produk** dalam konteks bisnis retail.

In [ ]:
# ============================================================
# STEP 1: INSTALL LIBRARY & MOUNT GOOGLE DRIVE
# ============================================================
# mlxtend tidak ada default di Colab, harus diinstall
# ============================================================

!pip install mlxtend -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd                           # Manipulasi data
import numpy as np                            # Operasi numerik
import matplotlib.pyplot as plt               # Visualisasi grafik
import seaborn as sns                         # Visualisasi lebih cantik
import os                                     # Operasi sistem file

from mlxtend.frequent_patterns import apriori           # Algoritma Apriori untuk frequent itemsets
from mlxtend.frequent_patterns import association_rules # Menghasilkan aturan asosiasi

# Pengaturan tampilan
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)

# Folder kerja di Google Drive
DRIVE_FOLDER = '/content/drive/MyDrive/data-mining'

print("✅ Google Drive ter-mount & semua library berhasil diimport!")
print(f"📁 Folder kerja: {DRIVE_FOLDER}")

In [ ]:
# ============================================================
# STEP 2: LOAD DATA BERSIH HASIL PREPROCESSING (Video 1)
# ============================================================
# File tersimpan di Google Drive
# ============================================================

data_path = os.path.join(DRIVE_FOLDER, 'online_retail_clean.csv')
df = pd.read_csv(data_path)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"✅ Data bersih berhasil dimuat! {len(df):,} baris x {df.shape[1]} kolom")
df.head()

## Step 3: Menyiapkan Data dalam Format Transaksi (Basket)

Algoritma Apriori membutuhkan data dalam format **basket** (keranjang belanja):
- Setiap baris = satu transaksi (InvoiceNo)
- Setiap kolom = satu produk (Description)
- Nilai = True/1 jika produk dibeli di transaksi tersebut, False/0 jika tidak

Format ini disebut **one-hot encoding** atau **transaction matrix**.

In [ ]:
# ============================================================
# MENYIAPKAN DATA DALAM FORMAT TRANSAKSI (BASKET)
# ============================================================
# Apriori membutuhkan format: setiap baris = 1 transaksi, kolom = produk
# Nilai = True jika produk dibeli di transaksi tersebut, False jika tidak
#
# Langkah:
# 1. Pastikan kolom Description bersih (hapus spasi berlebih)
# 2. Buat pivot table: baris=InvoiceNo, kolom=Description, nilai=Quantity
# 3. Konversi ke boolean: True jika Quantity > 0, False jika 0
# ============================================================

# Bersihkan kolom Description: hapus spasi di awal/akhir
# Ini penting agar nama produk yang sama tidak dianggap berbeda
df['Description'] = df['Description'].str.strip()

# Buat basket: pivot table dengan InvoiceNo sebagai baris, Description sebagai kolom
# Nilai = jumlah Quantity per transaksi per produk
# aggfunc='sum' → jumlahkan quantity jika ada duplikat
basket = df.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().fillna(0)

# Konversi ke boolean: True jika produk dibeli (Quantity > 0), False jika tidak
# Apriori hanya butuh info "dibeli atau tidak", bukan jumlahnya
def encode_units(x):
    """Fungsi helper: konversi quantity ke boolean"""
    return True if x > 0 else False

basket_bool = basket.applymap(encode_units)

print(f"✅ Format basket berhasil dibuat!")
print(f"   Jumlah transaksi (baris): {basket_bool.shape[0]:,}")
print(f"   Jumlah produk unik (kolom): {basket_bool.shape[1]:,}")
print(f"\n📋 Contoh data basket (5 baris pertama, 5 kolom pertama):")
basket_bool.iloc[:5, :5]

## Step 4: Menjalankan Algoritma Apriori — Menemukan Frequent Itemsets

**Konsep Apriori**:
- **Itemset** = kumpulan item/produk (misal: {Meja, Kursi})
- **Frequent Itemset** = itemset yang muncul di ≥ min_support transaksi
- **Support** = proporsi transaksi yang mengandung itemset tersebut
  - Rumus: Support(A) = jumlah transaksi mengandung A / total transaksi
- **min_support** = batas minimum support agar itemset dianggap "sering muncul"
  - Kita set 0.02 (2%) → itemset harus muncul di minimal 2% transaksi

In [ ]:
# ============================================================
# MENJALANKAN ALGORITMA APRIORI — FREQUENT ITEMSETS
# ============================================================
# apriori() menemukan semua itemset yang support-nya ≥ min_support
#
# Parameter:
#   basket_bool   → data dalam format basket (boolean)
#   min_support   → batas minimum support (0.02 = 2% transaksi)
#                    Artinya: itemset harus muncul di minimal 2% transaksi
#   use_colnames  → tampilkan nama produk (bukan index kolom)
#   verbose=0     → jangan tampilkan log detail
#
# Rumus Support:
#   Support(A) = jumlah transaksi mengandung A / total transaksi
#   Contoh: Support({Meja}) = 5% → Meja dibeli di 5% transaksi
# ============================================================

frequent_itemsets = apriori(
    basket_bool,          # Data basket format boolean
    min_support=0.02,     # Minimum support 2% (itemset muncul di ≥2% transaksi)
    use_colnames=True,    # Tampilkan nama produk, bukan index angka
    verbose=0             # Jangan tampilkan log proses
)

# Tambahkan kolom panjang itemset (jumlah item dalam itemset)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print(f"✅ Frequent Itemsets berhasil ditemukan!")
print(f"   Total itemsets: {len(frequent_itemsets):,}")
print(f"\n📋 Top 10 Frequent Itemsets (support tertinggi):")
frequent_itemsets.sort_values('support', ascending=False).head(10)

In [ ]:
# ============================================================
# LIHAT FREQUENT ITEMSETS DENGAN 2 ITEM (PASANGAN PRODUK)
# ============================================================
# Itemset dengan 2 item = pasangan produk yang sering dibeli bersamaan
# Ini yang paling berguna untuk membuat aturan asosiasi
# ============================================================

# Filter: hanya itemset dengan 2 item
pair_itemsets = frequent_itemsets[frequent_itemsets['length'] == 2].sort_values('support', ascending=False)

print(f"📋 Top 15 Pasangan Produk Paling Sering Dibeli Bersamaan:")
print("=" * 70)
for idx, row in pair_itemsets.head(15).iterrows():
    items = list(row['itemsets'])
    print(f"   {items[0][:40]:40s} ↔ {items[1][:40]:40s}  Support={row['support']:.4f}")

print(f"\n💡 Support 0.04 artinya: pasangan produk ini muncul di 4% dari seluruh transaksi")

## Step 5: Menghasilkan Aturan Asosiasi (Association Rules)

Dari frequent itemsets, kita menghasilkan **aturan asosiasi** berbentuk:
**"Jika beli A → maka cenderung beli B"** (A → B)

**Tiga metrik penting**:
- **Support**: seberapa sering aturan ini muncul di seluruh transaksi
  - Support(A→B) = P(A ∩ B) = jumlah transaksi mengandung A dan B / total transaksi
- **Confidence**: dari semua transaksi yang beli A, berapa persen yang juga beli B
  - Confidence(A→B) = P(B|A) = Support(A∩B) / Support(A)
  - Confidence tinggi → aturan ini cukup andal
- **Lift**: seberapa kuat hubungan A dan B dibandingkan jika keduanya independen
  - Lift(A→B) = Confidence(A→B) / Support(B) = P(A∩B) / (P(A) × P(B))
  - Lift > 1 → A dan B positif berkorelasi (saling mendukung)
  - Lift = 1 → A dan B independen (tidak ada hubungan)
  - Lift < 1 → A dan B negatif berkorelasi (saling menolak)

In [ ]:
# ============================================================
# MENGHASILKAN ATURAN ASOSIASI (ASSOCIATION RULES)
# ============================================================
# association_rules() mengubah frequent itemsets menjadi aturan A→B
#
# Parameter:
#   frequent_itemsets → hasil dari apriori()
#   metric='confidence' → gunakan confidence sebagai metrik filter utama
#   min_threshold=0.3 → hanya tampilkan aturan dengan confidence ≥ 30%
#     Artinya: dari transaksi yang beli A, minimal 30% juga beli B
#
# Metrik yang dihasilkan:
#   support    = P(A ∩ B) → seberapa sering A dan B muncul bersamaan
#   confidence = P(B|A)   → dari yang beli A, berapa persen juga beli B
#   lift       = P(A∩B)/(P(A)×P(B)) → kekuatan hubungan A dan B
#     lift > 1 = positif berkorelasi (saling mendukung) → ATURAN BAGUS
#     lift = 1 = independen (tidak ada hubungan)
#     lift < 1 = negatif berkorelasi (saling menolak)
# ============================================================

rules = association_rules(
    frequent_itemsets,       # Frequent itemsets dari apriori
    metric='confidence',     # Metrik filter utama: confidence
    min_threshold=0.3,       # Minimum confidence 30%
    num_itemsets=len(frequent_itemsets)  # Jumlah itemsets (parameter baru mlxtend)
)

# Tambahkan kolom panjang antecedent dan consequent untuk analisis
rules['antecedent_len'] = rules['antecedents'].apply(lambda x: len(x))
rules['consequent_len'] = rules['consequents'].apply(lambda x: len(x))

print(f"✅ Aturan asosiasi berhasil dihasilkan!")
print(f"   Total aturan: {len(rules):,}")
print(f"\n📋 10 Aturan Terbaik (berdasarkan Lift tertinggi):")
rules.sort_values('lift', ascending=False).head(10)

In [ ]:
# ============================================================
# FILTER ATURAN TERBAIK: LIFT > 1 DAN CONFIDENCE TINGGI
# ============================================================
# Kita hanya tertarik pada aturan yang:
#   1. Lift > 1 → A dan B positif berkorelasi (saling mendukung)
#   2. Confidence ≥ 0.4 → minimal 40% dari pembeli A juga beli B
#   3. Support ≥ 0.02 → aturan muncul di minimal 2% transaksi (cukup signifikan)
# ============================================================

# Filter aturan yang kuat
strong_rules = rules[
    (rules['lift'] > 1) &              # Positif berkorelasi
    (rules['confidence'] >= 0.4) &      # Confidence minimal 40%
    (rules['support'] >= 0.02)          # Support minimal 2%
].sort_values('lift', ascending=False)

print(f"📋 Aturan Asosiasi Kuat (Lift>1, Confidence≥40%, Support≥2%):")
print(f"   Jumlah aturan: {len(strong_rules)}")
print()

# Tampilkan top 15 aturan terkuat
print("🏆 Top 15 Aturan Terkuat (Lift Tertinggi):")
print("=" * 80)
for idx, row in strong_rules.head(15).iterrows():
    ant = ', '.join(list(row['antecedents']))[:35]
    con = ', '.join(list(row['consequents']))[:35]
    print(f"   {ant:35s} → {con:35s}")
    print(f"   Support={row['support']:.3f}  Confidence={row['confidence']:.3f}  Lift={row['lift']:.2f}")
    print()

In [ ]:
# ============================================================
# VISUALISASI: TOP 10 ATURAN BERDASARKAN LIFT
# ============================================================
# Scatter plot: Support vs Confidence, ukuran titik = Lift
# Semakin besar titik → Lift semakin tinggi → aturan semakin kuat
# ============================================================

fig, ax = plt.subplots(figsize=(10, 7))

# Ambil top 50 aturan untuk visualisasi
top_rules = strong_rules.head(50)

scatter = ax.scatter(
    top_rules['support'],           # Sumbu X: Support
    top_rules['confidence'],        # Sumbu Y: Confidence
    s=top_rules['lift'] * 50,      # Ukuran titik proporsional terhadap Lift
    c=top_rules['lift'],           # Warna berdasarkan Lift
    cmap='YlOrRd',                  # Colormap: kuning-oranye-merah
    alpha=0.7,
    edgecolors='grey',
    linewidth=0.5
)

# Colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Lift', fontsize=12)

ax.set_xlabel('Support', fontsize=13)
ax.set_ylabel('Confidence', fontsize=13)
ax.set_title('Aturan Asosiasi — Support vs Confidence (ukuran titik = Lift)',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("💡 Titik di kanan atas = aturan yang sering muncul DAN andal.")
print("   Titik besar (Lift tinggi) = hubungan A→B sangat kuat, bukan kebetulan.")

In [ ]:
# ============================================================
# VISUALISASI: HEATMAP CONFIDENCE TOP ATURAN
# ============================================================
# Heatmap menunjukkan confidence antara antecedent (sumbu Y) dan consequent (sumbu X)
# Warna lebih gelap = confidence lebih tinggi
# ============================================================

# Ambil top 15 aturan untuk heatmap
top15 = strong_rules.head(15).copy()

# Buat label singkat untuk antecedent dan consequent
top15['ant_label'] = top15['antecedents'].apply(lambda x: ', '.join(list(x))[:25])
top15['con_label'] = top15['consequents'].apply(lambda x: ', '.join(list(x))[:25])

# Buat pivot table untuk heatmap
heatmap_data = top15.pivot_table(
    index='ant_label',
    columns='con_label',
    values='confidence',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Confidence'})
ax.set_title('Heatmap Confidence — Top 15 Aturan Asosiasi', fontsize=14, fontweight='bold')
ax.set_xlabel('Consequent (Produk Akhir)', fontsize=12)
ax.set_ylabel('Antecedent (Produk Awal)', fontsize=12)
plt.tight_layout()
plt.show()

print("💡 Warna lebih gelap = confidence lebih tinggi.")
print("   Artinya: jika pelanggan beli produk di sumbu Y, kemungkinan besar juga beli produk di sumbu X.")

In [ ]:
# ============================================================
# VISUALISASI: PARALLEL COORDINATES — SUPPORT, CONFIDENCE, LIFT
# ============================================================
# Parallel coordinates menunjukkan hubungan ketiga metrik secara bersamaan
# Setiap garis = satu aturan asosiasi
# Garis yang naik dari Support ke Confidence ke Lift = aturan yang sangat kuat
# ============================================================

# Ambil top 30 aturan untuk visualisasi yang jelas
top30 = strong_rules.head(30).copy()

# Buat label untuk setiap aturan
top30['rule'] = top30['antecedents'].apply(lambda x: ', '.join(list(x))[:20]) + \
                ' → ' + \
                top30['consequents'].apply(lambda x: ', '.join(list(x))[:20])

fig, ax = plt.subplots(figsize=(12, 8))

# Plot parallel coordinates manual
for idx, row in top30.iterrows():
    ax.plot(['Support', 'Confidence', 'Lift'],
            [row['support'], row['confidence'], row['lift']],
            alpha=0.5, linewidth=1.5,
            color=plt.cm.YlOrRd(row['lift'] / top30['lift'].max()))

ax.set_ylabel('Nilai Metrik', fontsize=12)
ax.set_title('Parallel Coordinates — Top 30 Aturan Asosiasi', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("💡 Garis yang tinggi di ketiga sumbu = aturan yang sering muncul, andal, dan kuat hubungannya.")

In [ ]:
# ============================================================
# INTERPRETASI HASIL & REKOMENDASI BISNIS
# ============================================================

print("=" * 70)
print("📊 INTERPRETASI HASIL ASSOCIATION RULES (APRIORI)")
print("=" * 70)
print()
print("1. JUMLAH ATURAN KUAT DITEMUKAN:", len(strong_rules))
print("   → Aturan dengan Lift>1, Confidence≥40%, Support≥2%")
print()
print("2. METRIK UTAMA:")
print("   - Support: seberapa sering aturan muncul di seluruh transaksi")
print("   - Confidence: dari pembeli A, berapa persen juga beli B")
print("   - Lift > 1: A dan B saling mendukung (bukan kebetulan)")
print()
print("3. REKOMENDASI BISNIS RETAIL:")
print()
print("   a) BUNDLING PRODUK:")
print("      → Pasangan produk dengan Lift tinggi bisa dijadikan paket bundling")
print("      → Contoh: jika 'Produk A → Produk B' punya Lift=5,")
print("        buat paket bundling 'Produk A + B' dengan diskon khusus")
print()
print("   b) PENATAAN LAYOUT TOKO:")
print("      → Produk yang sering dibeli bersamaan ditempatkan berdekatan")
print("      → Memudahkan pelanggan menemukan produk pelengkap")
print()
print("   c) REKOMENDASI PRODUK (CROSS-SELLING):")
print("      → Saat pelanggan membeli produk A, tawarkan produk B")
print("      → Gunakan aturan dengan confidence tinggi sebagai dasar rekomendasi")
print()
print("   d) PROMOSI TARGETED:")
print("      → Berikan kupon diskon untuk produk B kepada pembeli produk A")
print("      → Fokus pada aturan dengan Lift > 3 (hubungan sangat kuat)")
print()
print("4. KELEBIHAN APRIORI:")
print("   - Mudah dipahami dan diinterpretasi")
print("   - Menghasilkan aturan yang actionable untuk bisnis")
print("   - Tidak memerlukan label/target (unsupervised)")
print()
print("5. KEKURANGAN APRIORI:")
print("   - Min_support terlalu rendah → terlalu banyak aturan (kurang bermakna)")
print("   - Min_support terlalu tinggi → aturan penting terlewat")
print("   - Komputasi berat untuk dataset sangat besar (itemset kombinasi eksponensial)")
print("   - Tidak menunjukkan kausalitas (hanya korelasi)")